# DSP DEBUG

In [ ]:
from commstools import Signal, load_npz
from commstools.backend import dispatch
from commstools.equalization import lms, block_lms
from commstools.impairments import compensate_iq_imbalance_lowdin
from commstools.timing import correct_timing, estimate_timing
from commstools.frequency import (
    correct_static_frequency_offset,
    find_bias_tone,
    correct_frequency_drift,
    correct_static_frequency_offset,
)
from commstools.recovery import correct_carrier_phase, recover_carrier_phase_bps

TX_PATH = "/home/lokgar/repos/trmhi304-p2p/waveforms/signal_16qam"
RX_PATH = "/home/lokgar/repos/trmhi304-p2p/captures/capture_16qam_intradyne_long.npy"


In [ ]:
import jax

jax.config.update("jax_enable_x64", True)

# --------------------------------------------------------------------------
# 1. Reference TX waveform, brought to the ADC sample rate (8 sps)
# --------------------------------------------------------------------------

tx = load_npz(TX_PATH)
_, xp, sp = dispatch(tx.samples)

digoff = tx.digital_frequency_offset  # +300 MHz baked at TX
tx.shift_frequency(-digoff)  # strip it from the reference
tx.resample(up=1, down=2)  # 4 GSps → 2 GSps  (16 sps → 8 sps)
tile_len_adc = tx.samples.shape[-1]  # one DAC buffer at the ADC rate

# --------------------------------------------------------------------------
# 2. Capture: form 2-pol complex, fix IQ imbalance.
# --------------------------------------------------------------------------
rx_raw = xp.load(RX_PATH)
rx_samples = xp.array(
    [
        rx_raw[0] + 1j * rx_raw[1],
        rx_raw[2] + 1j * rx_raw[3],
    ]
)
rx_samples = compensate_iq_imbalance_lowdin(xp.conj(rx_samples))

rx = Signal(
    samples=rx_samples,
    sampling_rate=tx.sampling_rate,
    symbol_rate=tx.symbol_rate,
    mod_scheme=tx.frame.payload_mod_scheme,
    mod_order=tx.frame.payload_mod_order,
    frame=tx.frame,
    pulse_shape=tx.pulse_shape,
)

rx.plot_psd(title="RX PSD before coarse FOE correction", nperseg=2**15, show=True)

# --------------------------------------------------------------------------
# 3. Coarse FOE on the FIRST tile — drives the timing step.
#    Use the bias-leakage tone in the PSD with log-parabolic sub-bin
#    interpolation.  One sub-bin pass is enough precision for timing's
#    phase coherence; per-segment refinement comes later.
# --------------------------------------------------------------------------
f_b_coarse = find_bias_tone(rx.samples[0, :tile_len_adc], rx.sampling_rate)
coarse_shift = f_b_coarse + digoff  # puts bias at -digoff, signal at 0
print(
    f"    bias-tone @ {f_b_coarse:+.3f} MHz, "
    f"applying shift = {coarse_shift / 1e6:+.3f} MHz"
)
rx.samples = correct_static_frequency_offset(rx.samples, rx.sampling_rate, coarse_shift)

rx.plot_psd(title="RX PSD after coarse FOE correction", nperseg=2**15, show=True)

# --------------------------------------------------------------------------
# 4. Timing — first peak, slice mode.  Restrict the cross-correlation
#    search to the first 2 tiles so estimate_timing's argmax lands on the
#    FIRST tile peak (otherwise the strongest of many near-identical
#    tile-peaks wins, typically pointing deep into the capture).
# --------------------------------------------------------------------------
coarse, fract = estimate_timing(
    rx.samples,
    tx.samples,
    sps=rx.sps,
    search_range=(0, tile_len_adc),
    threshold=3,
    debug_plot=True,
)
rx.samples = correct_timing(rx.samples, coarse, fract, mode="slice")

# --------------------------------------------------------------------------
# 5. Truncate to an integer number of frames after the slice.
#    Doing this *after* the slice (not before) preserves up to one extra
#    frame relative to a pre-trim, since the raw trailing partial tile
#    can now contribute when it's bigger than `max(coarse)`.
# --------------------------------------------------------------------------
n_seg_adc = rx.samples.shape[-1] // tile_len_adc
# rx.samples = rx.samples[:, : n_seg_adc * tile_len_adc]
rx.samples = rx.samples[:, : 10 * tile_len_adc]


# --------------------------------------------------------------------------
# 6. Per-segment refined FOE with a continuous-phase trajectory.
#    Search around the post-coarse bias-tone location (-digoff) in each
#    segment, compute the residual drift, cubic-spline-interpolate, and
#    integrate to a per-sample phase array applied via
#    correct_carrier_phase.  No phase discontinuities at boundaries.
# --------------------------------------------------------------------------
def per_segment_estimator(blk, fs):
    # After the coarse correction, the bias tone sits near -digoff.
    # Any deviation = the residual frequency drift to be removed.
    f_b_k = find_bias_tone(
        blk,
        fs,
        target_hz=-digoff,
        search_band_hz=20e6,  # ±20 MHz around -digoff is well clear of the signal
    )
    return f_b_k + digoff  # δf_k = current bias offset relative to target


rx.samples = correct_frequency_drift(
    rx.samples,
    rx.sampling_rate,
    block_size=tile_len_adc,
    overlap=0.5,
    estimator=per_segment_estimator,
    combine_channels=True,
    debug_plot=True,
)

# --------------------------------------------------------------------------
# 7. Resample reference and capture to 2 sps; the polyphase anti-alias
#    filter kills the now-stationary bias tone at -digoff = -300 MHz
#    (outside ±250 MHz Nyquist at 500 MSps).
# --------------------------------------------------------------------------
rx.resample(sps_out=2)
tx.resample(sps_out=2)
n_seg = rx.samples.shape[-1] // tx.samples.shape[-1]

rx.plot_psd(
    show=True,
    title="After FOE + timing + per-segment refinement (2 sps)",
    nperseg=2**15,
)

# --------------------------------------------------------------------------
# 8. Matched filter + tile source bits/symbols to the aligned frame count.
# --------------------------------------------------------------------------
rx.source_bits = xp.tile(tx.frame.payload_bits, (1, n_seg))
rx.source_symbols = xp.tile(tx.frame.payload_symbols, (1, n_seg))
rx.matched_filter()
rx.plot_psd(show=True, title="Synced and MFed PSD", nperseg=2**15)


## JAX LMS

In [ ]:
# --------------------------------------------------------------------------
# 9. Iterative warm-started butterfly LMS with BPS CPR.
#    Known limitation: CPR state resets to 0 on every block_lms entry.
# --------------------------------------------------------------------------
NUM_EQ = 4
NUM_TRAIN = 2**13
NUM_TAPS = 21
STEP_SIZE = 2e-3
BPS_TEST_PHASES = 256
BPS_BLOCK_SIZE = 22

eq_kwargs = dict(
    modulation=rx.mod_scheme,
    order=rx.mod_order,
    num_taps=NUM_TAPS,
    step_size=STEP_SIZE,
    sps=rx.sps,
    cpr_type="bps",
    cpr_joint_channels=True,
    cpr_bps_test_phases=BPS_TEST_PHASES,
    cpr_bps_block_size=BPS_BLOCK_SIZE,
    cpr_cycle_slip_correction=True,
    cpr_cycle_slip_history=128,
    cpr_cycle_slip_threshold=xp.pi / 4,
    plot_smoothing=500,
    backend="jax",
)

res = None
for i in range(NUM_EQ):
    res = lms(
        samples=rx.samples[:, : NUM_TRAIN * 2],
        training_symbols=rx.source_symbols[:, :NUM_TRAIN],
        w_init=res.weights if res is not None else None,
        # input_norm_factor=res.input_norm_factor if res is not None else None,
        # cpr_state=res.cpr_state if res is not None else None,
        debug_plot=(i == 0 or i == NUM_EQ - 1),
        **eq_kwargs,
    )

res = lms(
    samples=rx.samples,
    training_symbols=rx.source_symbols[:, :NUM_TRAIN],
    w_init=res.weights if res is not None else None,
    input_norm_factor=res.input_norm_factor if res is not None else None,
    # cpr_state=res.cpr_state if res is not None else None,
    debug_plot=True,
    **eq_kwargs,
)

eq = rx.copy()
eq.samples = res.y_hat
eq.sampling_rate = rx.symbol_rate
eq.plot_constellation(show=True, overlay_ideal=True)

# --------------------------------------------------------------------------
# 10. Standalone BPS-CPR on the equalized output, then phase ambiguity
#     resolution and hard demap.
# --------------------------------------------------------------------------
phases_bps = recover_carrier_phase_bps(
    eq.samples,
    modulation=eq.mod_scheme,
    order=eq.mod_order,
    debug_plot=True,
    block_size=BPS_BLOCK_SIZE,
    num_test_phases=BPS_TEST_PHASES,
    joint_channels=False,
)

rec = eq.copy()
rec.samples = correct_carrier_phase(rec.samples, phases_bps)
rec.resolve_symbols()
rec.resolve_phase_ambiguity()
rec.demap_symbols_hard()
rec.plot_constellation(show=True, overlay_ideal=True)

# --------------------------------------------------------------------------
# 11. Metrics.  EVM "blind" measures blob size against the nearest
#     constellation point.  SER/SNR discard `2 × NUM_TRAIN` symbols to
#     skip past the post-training CPR convergence region (Issue A).
# --------------------------------------------------------------------------
rec.evm(num_train_symbols=NUM_TRAIN, mode="blind")
rec.ser(num_train_symbols=NUM_TRAIN)
rec.snr(num_train_symbols=NUM_TRAIN)


## NUMBA LMS

In [ ]:
# --------------------------------------------------------------------------
# 9. Iterative warm-started butterfly LMS with BPS CPR.
#    Known limitation: CPR state resets to 0 on every block_lms entry.
# --------------------------------------------------------------------------
NUM_EQ = 4
NUM_TRAIN = 2**13
NUM_TAPS = 21
STEP_SIZE = 3e-3
BPS_TEST_PHASES = 256
BPS_BLOCK_SIZE = 20

eq_kwargs = dict(
    modulation=rx.mod_scheme,
    order=rx.mod_order,
    num_taps=NUM_TAPS,
    step_size=STEP_SIZE,
    sps=rx.sps,
    cpr_type="bps",
    cpr_joint_channels=True,
    cpr_bps_test_phases=BPS_TEST_PHASES,
    cpr_bps_block_size=BPS_BLOCK_SIZE,
    cpr_cycle_slip_correction=True,
    cpr_cycle_slip_history=128,
    cpr_cycle_slip_threshold=xp.pi / 4,
    plot_smoothing=500,
    backend="numba",
)

res = None
for i in range(NUM_EQ):
    res = lms(
        samples=rx.samples[:, : NUM_TRAIN * 2],
        training_symbols=rx.source_symbols[:, :NUM_TRAIN],
        w_init=res.weights if res is not None else None,
        # input_norm_factor=res.input_norm_factor if res is not None else None,
        # cpr_state=res.cpr_state if res is not None else None,
        debug_plot=(i == 0 or i == NUM_EQ - 1),
        **eq_kwargs,
    )

res = lms(
    samples=rx.samples,
    training_symbols=rx.source_symbols[:, :NUM_TRAIN],
    w_init=res.weights if res is not None else None,
    input_norm_factor=res.input_norm_factor if res is not None else None,
    # cpr_state=res.cpr_state if res is not None else None,
    debug_plot=True,
    **eq_kwargs,
)

eq = rx.copy()
eq.samples = res.y_hat
eq.sampling_rate = rx.symbol_rate
eq.plot_constellation(show=True, overlay_ideal=True)

# --------------------------------------------------------------------------
# 10. Standalone BPS-CPR on the equalized output, then phase ambiguity
#     resolution and hard demap.
# --------------------------------------------------------------------------
phases_bps = recover_carrier_phase_bps(
    eq.samples,
    modulation=eq.mod_scheme,
    order=eq.mod_order,
    debug_plot=True,
    block_size=BPS_BLOCK_SIZE,
    num_test_phases=BPS_TEST_PHASES,
    joint_channels=False,
)

rec = eq.copy()
rec.samples = correct_carrier_phase(rec.samples, phases_bps)
rec.resolve_symbols()
rec.resolve_phase_ambiguity()
rec.demap_symbols_hard()
rec.plot_constellation(show=True, overlay_ideal=True)

# --------------------------------------------------------------------------
# 11. Metrics.  EVM "blind" measures blob size against the nearest
#     constellation point.  SER/SNR discard `2 × NUM_TRAIN` symbols to
#     skip past the post-training CPR convergence region (Issue A).
# --------------------------------------------------------------------------
rec.evm(num_train_symbols=NUM_TRAIN, mode="blind")
rec.ser(num_train_symbols=NUM_TRAIN)
rec.snr(num_train_symbols=NUM_TRAIN)


## BLOCK-LMS

In [ ]:
# --------------------------------------------------------------------------
# 9. Iterative warm-started butterfly block-LMS with BPS CPR.
#    Known limitation: CPR state resets to 0 on every block_lms entry.
# --------------------------------------------------------------------------
NUM_EQ = 6
NUM_TRAIN = 2**13
NUM_TAPS = 25
BLOCK_SIZE = 1024
STEP_SIZE = 8e-4

eq_kwargs = dict(
    modulation=rx.mod_scheme,
    order=rx.mod_order,
    num_taps=NUM_TAPS,
    step_size=STEP_SIZE,
    sps=rx.sps,
    block_size=BLOCK_SIZE,
    cpr_type="bps",
    cpr_bps_joint_channels=True,
    cpr_bps_test_phases=128,
    cpr_bps_block_size=20,
    cpr_cycle_slip_correction=True,
    cpr_cycle_slip_history=128,
    cpr_cycle_slip_threshold=xp.pi / 4,
    plot_smoothing=500,
)

res = None
for i in range(NUM_EQ):
    res = block_lms(
        samples=rx.samples[:, : NUM_TRAIN * 2],
        training_symbols=rx.source_symbols[:, :NUM_TRAIN],
        w_init=res.weights if res is not None else None,
        debug_plot=(i == 0 or i == NUM_EQ - 1),
        **eq_kwargs,
    )

res = block_lms(
    samples=rx.samples,
    training_symbols=rx.source_symbols[:, :NUM_TRAIN],
    w_init=res.weights,
    input_norm_factor=res.input_norm_factor if res is not None else None,
    debug_plot=True,
    **eq_kwargs,
)

eq = rx.copy()
eq.samples = res.y_hat
eq.sampling_rate = rx.symbol_rate
eq.plot_constellation(show=True, overlay_ideal=True)


# --------------------------------------------------------------------------
# 10. Standalone BPS-CPR on the equalized output, then phase ambiguity
#     resolution and hard demap.
# --------------------------------------------------------------------------
phases_bps = recover_carrier_phase_bps(
    eq.samples,
    modulation=eq.mod_scheme,
    order=eq.mod_order,
    debug_plot=True,
    block_size=20,
    num_test_phases=128,
    joint_channels=False,
)

rec = eq.copy()
rec.samples = correct_carrier_phase(rec.samples, phases_bps)
rec.resolve_symbols()
rec.resolve_phase_ambiguity()
rec.demap_symbols_hard()
rec.plot_constellation(show=True, overlay_ideal=True)

# --------------------------------------------------------------------------
# 11. Metrics.  EVM "blind" measures blob size against the nearest
#     constellation point.  SER/SNR discard `2 × NUM_TRAIN` symbols to
#     skip past the post-training CPR convergence region (Issue A).
# --------------------------------------------------------------------------
rec.evm(num_train_symbols=NUM_TRAIN, mode="blind")
rec.ser(num_train_symbols=NUM_TRAIN)
rec.snr(num_train_symbols=NUM_TRAIN)
